# EcoHome Energy Advisor — RAG Pipeline Setup

This notebook:
1. Loads all energy-saving tip documents from `data/documents/`
2. Splits them into chunks for vector embedding
3. Creates OpenAI embeddings and stores them in a ChromaDB vector store
4. Verifies retrieval with sample queries

**Prerequisites:** Run `01_db_setup.ipynb` first and ensure your `OPENAI_API_KEY` is set in `.env`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv()

import chromadb
from langchain_openai import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter

DOCS_DIR       = 'data/documents'
VECTORSTORE_DIR = 'data/vectorstore'
COLLECTION_NAME = 'energy_tips'

os.makedirs(VECTORSTORE_DIR, exist_ok=True)
print('OpenAI API Key set:', bool(os.getenv('OPENAI_API_KEY')))

## 1. Load Documents

In [ ]:
from pathlib import Path

docs = []
for txt_file in sorted(Path(DOCS_DIR).glob('*.txt')):
    content = txt_file.read_text(encoding='utf-8')
    # Extract topic from first line
    first_line = content.split('\n')[0]
    topic = first_line.replace('TOPIC:', '').strip() if 'TOPIC:' in first_line else txt_file.stem
    docs.append({
        'content': content,
        'source': txt_file.name,
        'topic': topic,
    })
    print(f'  Loaded: {txt_file.name} ({len(content):,} chars)')

print(f'\n✅ Loaded {len(docs)} documents total.')

## 2. Chunk Documents

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=['\n\n', '\n', '. ', ' ', ''],
)

chunks, metadatas, ids = [], [], []

for doc_idx, doc in enumerate(docs):
    doc_chunks = splitter.split_text(doc['content'])
    for chunk_idx, chunk in enumerate(doc_chunks):
        chunks.append(chunk)
        metadatas.append({'source': doc['source'], 'topic': doc['topic'], 'chunk_idx': chunk_idx})
        ids.append(f"doc{doc_idx}_chunk{chunk_idx}")

print(f'✅ Created {len(chunks)} chunks across {len(docs)} documents.')
print(f'   Average chunk size: {sum(len(c) for c in chunks) // len(chunks)} chars')

## 3. Create Embeddings and Store in ChromaDB

In [ ]:
embeddings_model = OpenAIEmbeddings(
    model='text-embedding-3-small',
    openai_api_key=os.getenv('OPENAI_API_KEY'),
)

print('Computing embeddings (this may take 30-60 seconds)...')
embeddings = embeddings_model.embed_documents(chunks)
print(f'✅ Generated {len(embeddings)} embedding vectors (dim={len(embeddings[0])})')

In [ ]:
client = chromadb.PersistentClient(path=VECTORSTORE_DIR)

# Delete existing collection if re-running
try:
    client.delete_collection(COLLECTION_NAME)
    print('Deleted existing collection.')
except Exception:
    pass

collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={'hnsw:space': 'cosine'},
)

# Batch upsert (ChromaDB prefers batches of ≤5000)
BATCH = 500
for i in range(0, len(chunks), BATCH):
    collection.add(
        documents=chunks[i:i+BATCH],
        embeddings=embeddings[i:i+BATCH],
        metadatas=metadatas[i:i+BATCH],
        ids=ids[i:i+BATCH],
    )

print(f'✅ Stored {collection.count()} chunks in ChromaDB at: {VECTORSTORE_DIR}')

## 4. Verify Retrieval with Sample Queries

In [ ]:
TEST_QUERIES = [
    'When is the best time to charge my electric car?',
    'How can I optimise my HVAC to save money on electricity?',
    'What is the best battery storage strategy for time-of-use pricing?',
    'How do I maximise solar self-consumption during summer?',
    'What smart home automations can reduce my energy bill?',
]

for query in TEST_QUERIES:
    q_emb = embeddings_model.embed_query(query)
    results = collection.query(
        query_embeddings=[q_emb],
        n_results=2,
        include=['documents', 'metadatas', 'distances'],
    )
    print(f'\n🔍 Query: {query}')
    for doc, meta, dist in zip(results['documents'][0], results['metadatas'][0], results['distances'][0]):
        print(f"   [{meta['source']}] score={1-dist:.3f}")
        print(f"   → {doc[:150].strip()}...")

## 5. Summary

In [ ]:
print('=== RAG Pipeline Summary ===')
print(f'  Documents loaded:     {len(docs)}')
print(f'  Chunks created:       {len(chunks)}')
print(f'  Embeddings stored:    {collection.count()}')
print(f'  Embedding model:      text-embedding-3-small')
print(f'  Vector store:         ChromaDB (persistent)')
print(f'  Location:             {VECTORSTORE_DIR}')
print()
print('✅ RAG pipeline ready! Run 03_run_and_evaluate.ipynb next.')